# 08-13 · 추가 탐색 N1~N7 (36개)

| 항목 | 내용 |
|---|---|
| 하는 일 | 1차 탐색 뒤 계획서(`A_추가탐색계획_11-05.html`)에서 실행 전 고정한 36개 사양을 1회 실행해 `A_추가탐색결과_11-05.html`로 저장 |
| 읽는 파일 | `outputs/07_분석/패널_기업분기.csv`, `outputs/04_본수집/본수집_결과_기업별.csv`, 공통함수 `lib.py` |
| 만드는 파일 | `outputs/07_분석/모듈별/N_결과.csv`, `N_요약.txt`, `docs/A_추가탐색결과_11-05.html` |
| 인증키 | 필요 없음 (저장된 파일만 읽습니다) |
| 걸리는 시간 | 약 10초 — 2026-11-05, 이 컴퓨터에서 측정 |

> 이 노트북이 **원본**입니다. 같은 이름의 `.py`는 `scripts/노트북에서_py_만들기.py`가 여기서 자동으로 만든 파일이라 직접 고치지 않습니다. 원래 손으로 쓴 `.py`는 이 폴더의 `원본_스크립트/`에 보관했습니다.

판정 세부는 계획서 7절에 **실행 전** 고정했습니다. 이번 라운드를 끝으로 사양 탐색은 더 늘리지 않기로 했습니다(검증이 연구 질문보다 커지는 것을 막기 위함. 이 결정도 팀 확인 전).

> **성격** — 이 단계의 결과는 모두 **사후 탐색**입니다(결과를 본 뒤 설계가 바뀔 수 있고, 팀이 규칙을 확정하기 전입니다). '신호 기준'은 에이전트가 정한 임의 기준이며 확인 증거가 아닙니다.

### 준비 — 불러오기와 경로

필요한 도구를 불러오고, 이 폴더 위치(`HERE`)를 잡습니다. 맨 위 따옴표 글(독스트링)은 이 스크립트의 원래 사용설명서입니다.

In [1]:
"""추가 탐색 N1~N7 (docs/A_추가탐색계획_11-05.html, 판정 세부는 같은 문서 7절에 실행 전 고정). 36개를 1회 실행한다. 전부 사후 탐색.
실행: python -X utf8 13_추가탐색.py
"""
import csv
import html
import importlib.util
import sys
from pathlib import Path
import numpy as np
import pandas as pd
try:
    HERE = HERE      # .py로 실행할 때
except NameError:
    HERE = Path.cwd()                           # 노트북에서 실행할 때(노트북이 있는 폴더)
from lib import FE, LABEL, MOD, WORSE, add_T, config, finish, fit, load_panel, rows_from  # noqa: E402

### 상수 — 입력·출력 경로와 규칙 값

In [2]:
spec = importlib.util.spec_from_file_location("mods", HERE / "11_모듈.py")
mods = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mods)
if hasattr(sys.stdout, "reconfigure"):          # 노트북의 출력 창에는 이 함수가 없다
    sys.stdout.reconfigure(encoding="utf-8")
T3 = ["T1", "T2", "T3"]
DOCS = config.BASE / "docs"

#### `main()`

이 함수(132줄) 안의 단계 — 코드 주석에서 뽑은 목차:
1. N1, N2
2. N3 분위 비교 (사전ICR수준①)
3. N4 잔차 E
4. N5 이탈·ICR 미정의 기업 포함 (955곳)
5. N7 통제 후 위약
6. 판정 (계획서 7절)
7. HTML

In [3]:
def main():
    P = load_panel()
    S = mods.prep_S(P)

    def pre_rank(tmax, minn):
        d = S[S["t"] <= tmax].groupby("corp_code")["T3"].agg(["mean", "count"])
        return d[d["count"] >= minn]["mean"]

    S["pr"] = S["corp_code"].map(pre_rank(9, 6))
    S["pr19"] = S["corp_code"].map(pre_rank(3, 3))
    S["pr_x"] = S["pr"] * S["tight"]
    S["pr19_x"] = S["pr19"] * S["tight"]
    rows = []
    VAR = (("pr", "pr_x", "사전ICR수준①"), ("pr19", "pr19_x", "사전ICR수준②(2019만)"))
    # N1, N2
    for v, vx, lab in VAR:
        for y in T3:
            r = fit(S, y, ["Ex", vx], FE)
            rows += rows_from(r, "N", f"N1 {lab} 통제", ["E×Tight", lab + "×Tight"], "N12")
            r = fit(S, y, ["Ex", vx, "size_x", "lev_x", "roa_x"], FE)
            rows += rows_from(r, "N", f"N2 {lab}+사전특성 통제", ["E×Tight", lab + "×Tight", "ln자산×Tight", "레버리지×Tight", "영업이익/자산×Tight"], "N12")
    # N3 분위 비교 (사전ICR수준①)
    F = S.drop_duplicates("corp_code").dropna(subset=["pr"])
    q1, q2 = F["pr"].quantile([1 / 3, 2 / 3])
    S["pt"] = np.where(S["pr"] <= q1, 1, np.where(S["pr"] <= q2, 2, 3))
    for k in (1, 2, 3):
        S[f"Ex_p{k}"] = np.where(S["pr"].notna(), S["Ex"] * (S["pt"] == k), np.nan)
    for y in T3:
        r = fit(S, y, ["Ex_p1", "Ex_p2", "Ex_p3"], FE)
        rows += rows_from(r, "N", "N3 사전ICR수준① 3분위별 E×Tight", ["사전ICR 하위", "중간", "상위"], "N3", fam_all=True)
    # N4 잔차 E
    for v, vx, lab in VAR:
        Fv = S.drop_duplicates("corp_code").dropna(subset=["E_pre_A", "lev_pre", v, "roa_pre", "ln_assets_pre"])
        X = np.column_stack([np.ones(len(Fv)), Fv["lev_pre"], Fv[v], Fv["roa_pre"], Fv["ln_assets_pre"]])
        coef = np.linalg.lstsq(X, Fv["E_pre_A"].to_numpy(), rcond=None)[0]
        res = Fv["E_pre_A"].to_numpy() - X @ coef
        S["Ep2_x"] = S["corp_code"].map(dict(zip(Fv["corp_code"], res))) * S["tight"]
        for y in T3:
            r = fit(S, y, ["Ep2_x"], FE)
            rows += rows_from(r, "N", f"N4 잔차 E({lab}) ×Tight", ["E_perp2×Tight"], "N4")
    # N5 이탈·ICR 미정의 기업 포함 (955곳)
    S5 = add_T(P[(~P["혼재"]) & (P["E정의분기"] >= 8)].copy())
    for y in T3:
        rows += rows_from(fit(S5, y, ["Ex"], FE), "N", "N5 인상기 ICR 8개 이상 규칙 삭제(955곳)", ["E×Tight"], "N5")
    R5 = pd.read_csv(config.BASE / "outputs" / "04_본수집" / "본수집_결과_기업별.csv", dtype={"corp_code": str})
    R5["corp_code"] = R5["corp_code"].str.zfill(8)
    f5 = S5.drop_duplicates("corp_code").merge(R5[["corp_code", "마지막_자료"]], on="corp_code")
    f5["이탈후보"] = f5["마지막_자료"] != "2024Q3"
    ex = f5[f5["이탈후보"]]
    main_firms = set(S["corp_code"])
    desc = [f"N5 기술통계: 955곳 중 마지막 자료가 2024Q3 이전인 이탈 후보 {len(ex)}곳 (E_pre_A 평균 {ex['E_pre_A'].mean():.3f}, 중앙값 {ex['E_pre_A'].median():.3f}; "
            f"나머지 평균 {f5[~f5['이탈후보']]['E_pre_A'].mean():.3f}, 중앙값 {f5[~f5['이탈후보']]['E_pre_A'].median():.3f})",
            f"   그중 주 표본(789곳)에 포함된 이탈 후보 {int(ex['corp_code'].isin(main_firms).sum())}곳, 제외된 이탈 후보 {int((~ex['corp_code'].isin(main_firms)).sum())}곳",
            f"   인상기 ICR 관측이 0개인 기업 {int((f5['post_icr_n'] == 0).sum())}곳, 8개 미만 {int((f5['post_icr_n'] < 8).sum())}곳"]
    # N7 통제 후 위약
    tmap = S.groupby("t")["tight"].first().to_dict()
    Sp = S[S["t"] <= 9].copy()
    for t0, nm in ((4, "2020Q1"), (6, "2020Q3")):
        Sp["tp"] = Sp["t"].map(lambda t: tmap[t - t0 + 10] if t >= t0 else 0.0)
        Sp["Epl"] = Sp["E_pre_A"] * Sp["tp"]
        Sp["pr19_p"] = Sp["pr19"] * Sp["tp"]
        for y in T3:
            r = fit(Sp, y, ["Epl", "pr19_p"], FE)
            rows += rows_from(r, "N", f"N7 위약 가짜 긴축 {nm} (사전ICR②×Tight_p 통제)", ["E×Tight_p", "사전ICR수준②×Tight_p"], "N7")
    R = finish(rows, "N", "결과")
    key = R[(R["변수"].isin(["E×Tight", "E_perp2×Tight", "E×Tight_p", "사전ICR 하위", "중간", "상위"]))]

    # 판정 (계획서 7절)
    def sel(prefix, var="E×Tight"):
        d = R[(R["사양"].str.startswith(prefix)) & (R["변수"] == var)]
        return d
    n2a = sel("N2 사전ICR수준①").set_index("결과변수")
    n2b = sel("N2 사전ICR수준②").set_index("결과변수")
    dA = {y: int(np.sign(n2a.loc[y, "계수"]) * WORSE[y]) for y in T3}      # +1 악화, -1 개선
    dB = {y: int(np.sign(n2b.loc[y, "계수"]) * WORSE[y]) for y in T3}
    same = len(set(dA.values())) == 1 and len(set(dB.values())) == 1 and set(dA.values()) == set(dB.values())
    direction = list(dA.values())[0] if same else 0
    c1 = same
    c2 = int((n2a["Holm_p"] < 0.10).sum()) >= 2
    n3 = R[R["사양"].str.startswith("N3")]
    ok3 = 0
    for y in T3:
        g = n3[n3["결과변수"] == y]
        ok3 += int((np.sign(g["계수"]) * WORSE[y] == direction).sum()) >= 2
    c3 = same and ok3 >= 2
    n7 = R[R["사양"].str.startswith("N7") & (R["변수"] == "E×Tight_p")]
    bad = n7[(np.sign(n7["계수"]) * n7["결과변수"].map(WORSE) == direction) & (n7["p"] < 0.10)]
    c4 = same and len(bad) == 0
    lab = {1: "악화", -1: "개선"}
    if same and c2 and c3 and c4:
        verdict = "악화 신호 (가설 방향)" if direction == 1 else "개선이 통제 후에도 남음"
    else:
        verdict = "통제 후 신호 없음"
    L = []
    L.append("[N 결과: 핵심 계수만]")
    for r in key.itertuples():
        hp = "" if pd.isna(r.Holm_p) else f" Holm={r.Holm_p:.3f}"
        L.append(f"  {r.사양[:52]:52s} {LABEL.get(r.결과변수, r.결과변수)[:14]:14s} {str(r.변수)[:14]:14s} {r.계수: .4f} ({r.표준오차:.4f}) p={r.p:.3f}{hp} N={r.관측:,}")
    L += [""] + desc + [""]
    L.append("[판정 세부]")
    L.append(f"  N2① 방향: " + ", ".join(f"{LABEL[y]}→{lab[dA[y]]}" for y in T3) + " | N2② 방향: " + ", ".join(f"{LABEL[y]}→{lab[dB[y]]}" for y in T3))
    L.append(f"  조건① N2 ①·② 방향이 모두 같다: {'예' if c1 else '아니오'}")
    L.append(f"  조건② N2①의 2개 이상이 Holm p<0.10: {'예' if c2 else '아니오'} (Holm p: " + ", ".join(f"{LABEL[y]} {n2a.loc[y,'Holm_p']:.3f}" for y in T3) + ")")
    L.append(f"  조건③ N3: 결과변수 3개 중 {ok3}개에서 3분위 중 2개 이상이 같은 방향: {'예' if c3 else '아니오'}")
    L.append(f"  조건④ N7: 같은 방향·p<0.10 위약 {len(bad)}개: {'예(없음)' if c4 else '아니오'}")
    L.append(f"  → 판정: {verdict}")
    text = "\n".join(L)
    print(text)
    (MOD / "N_요약.txt").write_text(text, encoding="utf-8")

    # HTML
    def tab(df, cols):
        h = "<thead><tr>" + "".join(f"<th>{c}</th>" for c in cols) + "</tr></thead>"
        b = "".join("<tr>" + "".join(f"<td>{html.escape(str(r[c]))}</td>" for c in cols) + "</tr>" for _, r in df.iterrows())
        return f"<div class='tw'><table>{h}<tbody>{b}</tbody></table></div>"
    d = R.copy()
    for c in ("계수", "표준오차"):
        d[c] = d[c].map(lambda x: "" if pd.isna(x) else f"{x:.4f}")
    for c in ("p", "Holm_p"):
        d[c] = d[c].map(lambda x: "" if pd.isna(x) else f"{x:.3f}")
    d["결과변수"] = d["결과변수"].map(lambda y: LABEL.get(y, y))
    cols = ["사양", "결과변수", "변수", "계수", "표준오차", "p", "Holm_p", "방향", "관측", "기업"]
    css = """<style>:root{--ink:#1d2a33;--line:#dde4e8;--bg:#f4f6f7;--paper:#fff;--amber:#8a5a00;--ambbg:#fff4dc;--code:#eef2f4}
@media(prefers-color-scheme:dark){:root:not([data-theme="light"]){--ink:#e6edf1;--line:#33424c;--bg:#12191e;--paper:#1a242b;--amber:#f0c36a;--ambbg:#3a3220;--code:#243139}}
body{margin:0;background:var(--bg);color:var(--ink);font:15px/1.7 'Malgun Gothic',sans-serif}.wrap{max-width:1000px;margin:0 auto;padding:18px 14px 60px}
section{background:var(--paper);border:1px solid var(--line);border-radius:12px;padding:16px 20px;margin-top:14px}.tw{overflow-x:auto}table{border-collapse:collapse;width:100%;font-size:12.5px;min-width:640px}th{background:var(--code);text-align:left}th,td{border-bottom:1px solid var(--line);padding:5px 8px}.box{background:var(--ambbg);border-left:4px solid var(--amber);border-radius:8px;padding:10px 14px;margin:10px 0}pre{background:var(--code);padding:10px 12px;border-radius:8px;overflow-x:auto;font-size:12.5px}</style>"""
    page = (f"<!doctype html><html lang='ko'><head><meta charset='utf-8'><meta name='viewport' content='width=device-width,initial-scale=1'><title>A 추가 탐색 결과</title>{css}</head><body><div class='wrap'>"
            "<h1>A 추가 탐색 결과 (사후 탐색 3차, 2026-11-05)</h1>"
            "<div class='box'><b>지위.</b> A_추가탐색계획_11-05.html에서 실행 전에 고정한 36개와 판정 세부를 그대로 1회 실행했습니다. 이전 결과를 본 뒤의 설계인 사후 탐색이며 확인 증거가 아닙니다. <b>이번이 마지막 사양 탐색 라운드</b>입니다.</div>"
            f"<section><h2>판정: {verdict}</h2><pre>{html.escape(chr(10).join(L[len(key)+1:]))}</pre></section>"
            f"<section><h2>전체 36개 사양 (모든 계수)</h2>{tab(d, cols)}</section></div></body></html>")
    (DOCS / "A_추가탐색결과_11-05.html").write_text(page, encoding="utf-8")

### (.py에만 들어가는 부분)

노트북에서는 `__name__`이 항상 `"__main__"`이라 아래 코드를 일반 셀로 두면 열 때마다 실행됩니다. 그래서 실행하지 않는 설명 셀에 두고, 변환 스크립트가 `.py`에만 넣습니다.

```python
if __name__ == "__main__":
    main()
```

In [4]:
main()

[N 결과: 핵심 계수만]
  N1 사전ICR수준① 통제                                       asinh(ICR)     E×Tight        -0.0738 (0.2035) p=0.717 Holm=1.000 N=12,740
  N2 사전ICR수준①+사전특성 통제                                  asinh(ICR)     E×Tight        -0.0263 (0.3299) p=0.936 Holm=1.000 N=12,740
  N1 사전ICR수준① 통제                                       1{ICR<1}       E×Tight         0.0210 (0.0431) p=0.627 Holm=1.000 N=12,740
  N2 사전ICR수준①+사전특성 통제                                  1{ICR<1}       E×Tight        -0.0302 (0.0924) p=0.744 Holm=1.000 N=12,740
  N1 사전ICR수준① 통제                                       ICR 업종×분기 순위   E×Tight        -0.0021 (0.0254) p=0.935 Holm=1.000 N=12,722
  N2 사전ICR수준①+사전특성 통제                                  ICR 업종×분기 순위   E×Tight        -0.0003 (0.0461) p=0.995 Holm=1.000 N=12,722
  N1 사전ICR수준②(2019만) 통제                                asinh(ICR)     E×Tight         0.0274 (0.2136) p=0.898 Holm=1.000 N=12,731
  N2 사전ICR수준②(2019만)+사전특성 통제                           asinh(ICR)     E×Tig

### 저장된 결과 미리보기 (맨 위 5행)

위 실행이 만들었거나 이미 저장돼 있던 결과 파일을 표로 엽니다. 파일 전체는 엑셀로 여세요.

In [5]:
import pandas as pd
from IPython.display import display

_p = HERE.parents[1] / "outputs/07_분석/모듈별/N_결과.csv"
print(_p.name, "|", end=" ")
_df = pd.read_csv(_p, encoding="utf-8-sig", low_memory=False)
print(f"{len(_df):,}행 × {_df.shape[1]}열")
display(_df.head(5))

N_결과.csv | 72행 × 13열


,모듈,사양,결과변수,변수,계수,표준오차,p,관측,기업,가족,비고,Holm_p,방향
0,N,N1 사전ICR수준① 통제,T1,E×Tight,-0.073788,0.203507,7.170470e-01,12740,590,N12,NaN,1.0,악화
1,N,N1 사전ICR수준① 통제,T1,사전ICR수준①×Tight,-1.078953,0.109339,2.332722e-21,12740,590,NaN,NaN,NaN,악화
2,N,N2 사전ICR수준①+사전특성 통제,T1,E×Tight,-0.026322,0.329944,9.364408e-01,12740,590,N12,NaN,1.0,악화
3,N,N2 사전ICR수준①+사전특성 통제,T1,사전ICR수준①×Tight,-1.092208,0.144395,1.509906e-13,12740,590,NaN,NaN,NaN,악화
4,N,N2 사전ICR수준①+사전특성 통제,T1,ln자산×Tight,0.013006,0.015718,4.083007e-01,12740,590,NaN,NaN,NaN,개선
